In [15]:
"""
Computational Electrocatalysis: Water Splitting Project
"""

import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
from pyscf import gto, scf, mp, cc, dft
import contextlib
import io
import logging

try:
    from pyscf.geomopt.geometric_solver import optimize
except ImportError:
    print("Warning: geomeTRIC not found. Geometry optimization will be skipped.")
    optimize = lambda mf: mf.mol # Fallback if geomeTRIC is missing

def optimize_quietly(mf):
    """Runs PySCF geometry optimization while completely suppressing geomeTRIC logs."""
    f = io.StringIO()
    # Redirect both stdout and stderr
    with contextlib.redirect_stdout(f), contextlib.redirect_stderr(f):
        # Mute geomeTRIC's internal logger
        logger = logging.getLogger()
        old_level = logger.level
        logger.setLevel(logging.CRITICAL)
        try:
            opt_mol = optimize(mf)
        finally:
            # Restore the logger level once finished
            logger.setLevel(old_level)
    return opt_mol


# --- 1. Helper Functions for Chemistry & Geometry ---
BOHR_TO_ANGSTROM = 0.529177210903

def build_mol(coords, basis, spin=0):
    mol = gto.M(atom=coords, basis=basis, spin=spin, symmetry=False, verbose=0)
    return mol

def get_bond_length(mol, idx1, idx2):
    # PySCF returns coordinates in Bohr
    coords = mol.atom_coords() * BOHR_TO_ANGSTROM
    return np.linalg.norm(coords[idx1] - coords[idx2])

def get_bond_angle(mol, idx_a, idx_vertex, idx_c):
    coords = mol.atom_coords() * BOHR_TO_ANGSTROM
    v1 = coords[idx_a] - coords[idx_vertex]
    v2 = coords[idx_c] - coords[idx_vertex]
    cosine_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    return np.degrees(np.arccos(np.clip(cosine_angle, -1.0, 1.0)))

def run_calculation(mol, method="HF", xc="pbe"):
    if method == "HF":
        mf = scf.UHF(mol) if mol.spin > 0 else scf.RHF(mol)
    elif method == "DFT":
        mf = dft.UKS(mol) if mol.spin > 0 else dft.RKS(mol)
        mf.xc = xc
    mf.kernel()
    return mf

# --- 2. Widget Definitions & UI Layout ---
out = widgets.Output()

# Global Inputs
basis_dropdown = widgets.Dropdown(options=['sto-3g', '3-21g', '6-31g', 'cc-pvdz', 'cc-pvtz'], value='sto-3g', description='Base Basis:')
h2_coords = widgets.Text(value='H 0 0 0; H 0 0 0.74', description='H2 Coords:')
o2_coords = widgets.Text(value='O 0 0 0; O 0 0 1.21', description='O2 Coords:')
h2o_coords = widgets.Text(value='O 0 0 0; H 0 0.757 0.586; H 0 -0.757 0.586', description='H2O Coords:')

# Task specific triggers
btn_task345 = widgets.Button(description="Run Tasks 3, 4 & 5 (Geom & Energy)", button_style='info', layout=widgets.Layout(width='auto'))
btn_task67 = widgets.Button(description="Run Tasks 6 & 7 (Convergence)", button_style='warning', layout=widgets.Layout(width='auto'))
btn_task8 = widgets.Button(description="Run Task 8 (Methods)", button_style='danger', layout=widgets.Layout(width='auto'))

# Thermo inputs for Task 10
thermo_corr = widgets.FloatText(value=0.0, description='ΔG_corr (eV):', tooltip='Input NIST thermodynamic corrections (ZPE, H, -TS) in eV')
btn_task10 = widgets.Button(description="Run Task 10 (Thermodynamics)", button_style='success', layout=widgets.Layout(width='auto'))

# --- 3. Execution Logic ---
def execute_tasks_345(b):
    with out:
        clear_output()
        print("--- Running Tasks 3, 4 & 5: Geometry Optimization & SCF Energies ---")
        basis = basis_dropdown.value
        
        # Build
        m_h2 = build_mol(h2_coords.value, basis, spin=0)
        m_o2 = build_mol(o2_coords.value, basis, spin=2)
        m_h2o = build_mol(h2o_coords.value, basis, spin=0)
        
        # Optimize
        print("Optimizing geometries...")
        m_h2_opt = optimize_quietly(scf.RHF(m_h2))
        m_o2_opt = optimize_quietly(scf.UHF(m_o2))
        m_h2o_opt = optimize_quietly(scf.RHF(m_h2o))
        
        # Structural Data (Task 3)
        print("\n--- Task 3: Geometry & Structural Data ---")
        print(f"H-H Bond Length : {get_bond_length(m_h2_opt, 0, 1):.4f} Å")
        print(f"O-O Bond Length : {get_bond_length(m_o2_opt, 0, 1):.4f} Å")
        print(f"H2O O-H1 Length : {get_bond_length(m_h2o_opt, 0, 1):.4f} Å")
        print(f"H2O O-H2 Length : {get_bond_length(m_h2o_opt, 0, 2):.4f} Å")
        print(f"H2O H-O-H Angle : {get_bond_angle(m_h2o_opt, 1, 0, 2):.2f}°")
        
        # Energies (Tasks 4 & 5)
        print("\n--- Tasks 4 & 5: Electronic Energy Data ---")
        e_h2 = scf.RHF(m_h2_opt).kernel()
        e_o2 = scf.UHF(m_o2_opt).kernel()
        e_h2o = scf.RHF(m_h2o_opt).kernel()
        
        e_r_hartree = (2 * e_h2 + e_o2) - (2 * e_h2o)
        e_r_ev = e_r_hartree * 27.2114
        
        print(f"E_tot (H2)  : {e_h2:.6f} Hartree")
        print(f"E_tot (O2)  : {e_o2:.6f} Hartree")
        print(f"E_tot (H2O) : {e_h2o:.6f} Hartree")
        print(f"Reaction Energy (ΔE_r): {e_r_hartree:.6f} Hartree | {e_r_ev:.4f} eV")

def execute_tasks_67(b):
    with out:
        clear_output()
        print("--- Running Tasks 6 & 7: Basis Set Convergence ---")
        basis_series = ['sto-3g', '3-21g', '6-31g', 'cc-pvdz'] 
        print(f"{'Basis Set':<12} | {'E_tot H2O (Hartree)':<20} | {'ΔE_r (eV)':<10}")
        print("-" * 50)
        
        for b_set in basis_series:
            m_h2 = build_mol(h2_coords.value, b_set, spin=0)
            m_o2 = build_mol(o2_coords.value, b_set, spin=2)
            m_h2o = build_mol(h2o_coords.value, b_set, spin=0)
            
            e_h2 = scf.RHF(m_h2).kernel()
            e_o2 = scf.UHF(m_o2).kernel()
            e_h2o = scf.RHF(m_h2o).kernel()
            
            e_r_ev = ((2 * e_h2 + e_o2) - (2 * e_h2o)) * 27.2114
            print(f"{b_set:<12} | {e_h2o:<20.6f} | {e_r_ev:<10.4f}")

def execute_task_8(b):
    with out:
        clear_output()
        print("--- Running Task 8: Method Comparison ---")
        basis = basis_dropdown.value
        m_h2 = build_mol(h2_coords.value, basis, spin=0)
        m_o2 = build_mol(o2_coords.value, basis, spin=2)
        m_h2o = build_mol(h2o_coords.value, basis, spin=0)
        
        methods = ["HF", "MP2", "CCSD", "DFT-PBE", "DFT-PBE0"]
        print(f"{'Method':<12} | {'ΔE_r (eV)':<10}")
        print("-" * 25)
        
        # HF Baseline
        mf_h2 = scf.RHF(m_h2).run()
        mf_o2 = scf.UHF(m_o2).run()
        mf_h2o = scf.RHF(m_h2o).run()
        
        for mod in methods:
            try:
                if mod == "HF":
                    eh2, eo2, eh2o = mf_h2.e_tot, mf_o2.e_tot, mf_h2o.e_tot
                elif mod == "MP2":
                    eh2 = mp.MP2(mf_h2).run().e_tot
                    eo2 = mp.UMP2(mf_o2).run().e_tot
                    eh2o = mp.MP2(mf_h2o).run().e_tot
                elif mod == "CCSD":
                    eh2 = cc.CCSD(mf_h2).run().e_tot
                    eo2 = cc.UCCSD(mf_o2).run().e_tot
                    eh2o = cc.CCSD(mf_h2o).run().e_tot
                elif mod == "DFT-PBE":
                    eh2 = run_calculation(m_h2, "DFT", "pbe").e_tot
                    eo2 = run_calculation(m_o2, "DFT", "pbe").e_tot
                    eh2o = run_calculation(m_h2o, "DFT", "pbe").e_tot
                elif mod == "DFT-PBE0":
                    eh2 = run_calculation(m_h2, "DFT", "pbe0").e_tot
                    eo2 = run_calculation(m_o2, "DFT", "pbe0").e_tot
                    eh2o = run_calculation(m_h2o, "DFT", "pbe0").e_tot

                er_ev = ((2 * eh2 + eo2) - (2 * eh2o)) * 27.2114
                print(f"{mod:<12} | {er_ev:<10.4f}")
            except Exception as e:
                print(f"{mod:<12} | Failed: {e}")

def execute_task_10(b):
    with out:
        clear_output()
        print("--- Running Task 10: Final Accuracy Data ---")
        basis = basis_dropdown.value
        m_h2 = build_mol(h2_coords.value, basis, spin=0)
        m_o2 = build_mol(o2_coords.value, basis, spin=2)
        m_h2o = build_mol(h2o_coords.value, basis, spin=0)
        
        # Getting basic SCF energy for the equation
        e_h2 = scf.RHF(m_h2).kernel()
        e_o2 = scf.UHF(m_o2).kernel()
        e_h2o = scf.RHF(m_h2o).kernel()
        
        e_r_ev = ((2 * e_h2 + e_o2) - (2 * e_h2o)) * 27.2114
        
        # Apply Thermodynamics
        g_corr = thermo_corr.value
        delta_g = e_r_ev + g_corr
        reference_g = 4.92
        deviation = delta_g - reference_g
        
        print(f"Calculated ΔE_r           : {e_r_ev:.4f} eV")
        print(f"Thermodynamic Correction  : {g_corr:.4f} eV")
        print(f"Final Calculated ΔG_r     : {delta_g:.4f} eV")
        print("-" * 40)
        print(f"Experimental Reference    : {reference_g:.4f} eV")
        print(f"Absolute Deviation        : {abs(deviation):.4f} eV")

# --- 4. Event Binding & Display ---
btn_task345.on_click(execute_tasks_345)
btn_task67.on_click(execute_tasks_67)
btn_task8.on_click(execute_task_8)
btn_task10.on_click(execute_task_10)

ui = widgets.VBox([
    widgets.HTML("<h2>Electrocatalysis Project Dashboard</h2>"),
    widgets.HBox([basis_dropdown]),
    widgets.HBox([h2_coords, o2_coords, h2o_coords]),
    widgets.HTML("<hr>"),
    widgets.HBox([btn_task345, btn_task67, btn_task8]),
    widgets.HBox([thermo_corr, btn_task10]),
    widgets.HTML("<hr>"),
    out
])

display(ui)